# 01. Prepare data

Load filtered ECAD stations, segment into wet/dry spells per station, tag with dates and seasons, and dump to JSON used by notebooks 02–06.

**Inputs:** `data/ecad_data/ecad_data_south_europe_filtered/` and `data/ecad_data/all_stations_metadata_filtered.csv`.

**Output:** `one_sided_reflected_geometric_brownian_motion/notebooks/data/exports_json/` (kept in the legacy location so R scripts still pick it up).

Run once. The following notebooks load the JSONs directly.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

from article_code.util_files import config
from article_code.util_files import data_load as dl


# Config params
print('ECAD raw dir:', config.ECAD_RAW_DIR)
print('Exports JSON dir:', config.EXPORTS_JSON_DIR)
print('Figures dir:', config.FIGURES_DIR)
print('Threshold wet day (in 0.1mm):', config.WET_DAY_THRESHOLD)
print('Data start year:', config.START_YEAR)
print('Drops first and last spell (avoid incomplete spells):', config.DROP_FIRST_LAST)

ECAD raw dir: C:\Users\antoi\Desktop\code\companion_code_article\data\ecad_data\ecad_data_south_europe_filtered
Exports JSON dir: C:\Users\antoi\Desktop\code\companion_code_article\data\ecad_data\exports_json
Figures dir: C:\Users\antoi\Desktop\code\companion_code_article\figures\PALERMO
Threshold wet day (in 0.1mm): 6
Data start year: 1946
Drops first and last spell (avoid incomplete spells): True


In [2]:
# Load all stations. `load_all_data` returns a list of (name, processed_data) pairs.

dict_feature_eng = {
    "wet_day_threshold": config.WET_DAY_THRESHOLD,
    "drop_first_last": config.DROP_FIRST_LAST,
    "min_number_spells":config.MIN_NUMBER_SPELL,
    'start_year': config.START_YEAR
}

d = dl.load_all_data(dict_feature_eng=dict_feature_eng,
    verbose=False,
)

Loading stations: 100%|██████████| 247/247 [04:02<00:00,  1.02it/s, value=VALLE DE HECHO  HECHO -- 273342]        

Finished data load -- Rejected 38 dataframes


In [ ]:
# Extract spells (dry / wet) with start dates, then export in json format
base_filename = f"ecad_data_south_europe_filtered_after_{config.START_YEAR}_wet_day_thresh_{config.WET_DAY_THRESHOLD}"
filtered = dl.build_spells_export_filtered(d,
    out_dir=config.EXPORTS_JSON_DIR,
    base_filename=base_filename) # For file name: the threshold is applied earlier in load_data

### Fit dry spells: pwm method

In [3]:
# Load data
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))
import json
from article_code.util_files import config
from article_code.util_files.egp_pwm import *

base_filename = f"ecad_data_south_europe_filtered_after_{config.START_YEAR}_wet_day_thresh_{config.WET_DAY_THRESHOLD}.json"
json_path = config.EXPORTS_JSON_DIR / base_filename
# with open(json_path, encoding="utf-8") as f:
#     data = json.load(f)

# print(data['TOULOUSE-BLAGNAC']['dry_spell']['duration_spell'][:5])
# print(data['TOULOUSE-BLAGNAC']['dry_spell']['start_date_spell'][:5])

In [ ]:
# Fit dry spell duration with hdeGPD and save in ```name_folder_save_results```
subset_city_to_fit = False
name_folder_save_results = (
    f"fit_south_europe_subset_excess_over_"
    f"{config.WET_DAY_THRESHOLD}")
path_folder_save_results = config.RESULTS_FIT_DIR / name_folder_save_results

fit_hdegpd_dry_spell_durations(
    json_path=json_path,
    path_folder_save_results=path_folder_save_results,
    subset_city_to_fit=False)

### Fit wet spell: EM algorithm

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))
import json
from article_code.util_files import config
from article_code.util_files.mixt_geom_em import *

base_filename = f"ecad_data_south_europe_filtered_after_{config.START_YEAR}_wet_day_thresh_{config.WET_DAY_THRESHOLD}.json"
json_path = config.EXPORTS_JSON_DIR / base_filename
# with open(json_path, encoding="utf-8") as f:
#     data = json.load(f)
# print(data['TOULOUSE-BLAGNAC']['wet_spell']['duration_spell'][:5])
# print(data['TOULOUSE-BLAGNAC']['wet_spell']['start_date_spell'][:5])

In [ ]:
name_folder_save_results = (
    f"fit_south_europe_subset_excess_over_"
    f"{config.WET_DAY_THRESHOLD}")
path_folder_save_results = config.RESULTS_FIT_DIR / name_folder_save_results
fit_mixt_geom_wet_spell_durations(json_path,
    path_folder_save_results)
    